In [ ]:
# !pip install grpcio==1.42.0 tensorflow-serving-api==2.7.0

In [ ]:
# !pip install keras-image-helper

In [6]:
import grpc
import tensorflow as tf
from tensorflow_serving.apis import predict_pb2
from tensorflow_serving.apis import prediction_service_pb2_grpc

In [7]:
host = 'localhost:8500'

channel = grpc.insecure_channel(host)

stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)

In [8]:
from keras_image_helper import create_preprocessor

In [9]:
preprocessor = create_preprocessor('xception', target_size=(299,299))

In [11]:
url = "http://bit.ly/mlbookcamp-pants"
X = preprocessor.from_url(url)

In [ ]:
X

In [13]:
def np_to_protobuf(data):
    return tf.make_tensor_proto(X, shape=data.shape)

In [ ]:
np_to_protobuf(X)

In [15]:
pb_request = predict_pb2.PredictRequest()

pb_request.model_spec.name = 'clothing-model'
pb_request.model_spec.signature_name = 'serving_default'

pb_request.inputs['input_8'].CopyFrom(np_to_protobuf(X))

In [ ]:
pb_request

In [18]:
pb_response = stub.Predict(pb_request, timeout=20.0)

In [ ]:
pb_response

In [20]:
pb_response.outputs['dense_7'].float_val

[-1.8798640966415405, -4.756312370300293, -2.3595328330993652, -1.0892646312713623, 9.90378475189209, -2.826181173324585, -3.6483113765716553, 3.2411553859710693, -2.612095355987549, -4.852035999298096]

In [22]:
preds = pb_response.outputs['dense_7'].float_val

In [23]:
classes = [
    'dress',
    'hat',
    'longsleeve',
    'outwear',
    'pants',
    'shirt',
    'shoes',
    'shorts',
    'skirt',
    't-shirt'
]

In [ ]:
dict(zip(classes, preds))

{'dress': -1.8798640966415405,
 'hat': -4.756312370300293,
 'longsleeve': -2.3595328330993652,
 'outwear': -1.0892646312713623,
 'pants': 9.90378475189209,
 'shirt': -2.826181173324585,
 'shoes': -3.6483113765716553,
 'shorts': 3.2411553859710693,
 'skirt': -2.612095355987549,
 't-shirt': -4.852035999298096}